# Solutions · 08 · Linear least squares: calibrating a thermocouple

Worked solutions to the exercises of [notebook 08](../notebooks/08_linear_regression.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import datasets, fitting
cal = np.loadtxt(datasets.path("thermocouple_calibration.csv"), delimiter=",", skiprows=1)
T_ref, E = cal[:, 0], cal[:, 1]
X2 = np.column_stack([np.ones_like(T_ref), T_ref, T_ref**2])
quad = fitting.linear_lstsq(X2, E)

## Exercise 1

Fit a cubic. Is the cubic term significant? What happens to the confidence intervals of the other
   coefficients, and why?

In [2]:
X3 = np.column_stack([X2, T_ref**3])
cub = fitting.linear_lstsq(X3, E)
ci3 = cub.conf_int()
print(f"cubic term {cub.coef[3]:.2e}, 95 % CI [{ci3[3,0]:.2e}, {ci3[3,1]:.2e}]")
for i, name in enumerate(["a0", "a1", "a2"]):
    w2 = np.ptp(quad.conf_int()[i]); w3 = np.ptp(ci3[i])
    print(f"{name}: CI width quadratic fit {w2:.2e}, cubic fit {w3:.2e} ({w3/w2:.1f}x wider)")

cubic term 2.31e-09, 95 % CI [-2.42e-09, 7.03e-09]
a0: CI width quadratic fit 3.61e-02, cubic fit 4.13e-02 (1.1x wider)
a1: CI width quadratic fit 4.21e-04, cubic fit 9.54e-04 (2.3x wider)
a2: CI width quadratic fit 1.01e-06, cubic fit 5.76e-06 (5.7x wider)


The confidence interval of the cubic term contains zero: it is not supported by the data. Adding it also
**widens** the intervals of the other coefficients, because $T^2$ and $T^3$ are strongly correlated over
0-400 °C - the data cannot tell their effects apart, so each becomes less certain. Unnecessary terms
cost precision; keep the simplest model the data support.

## Exercise 2

Refit the quadratic with only the points at or below 200 °C and use it to predict the voltage at
   400 °C. Compare with the full fit: the danger of extrapolation.

In [3]:
low = T_ref <= 200
q_low = fitting.linear_lstsq(X2[low], E[low])
x400 = np.array([1, 400, 400**2])
print(f"voltage at 400 °C: measured {E[-1]:.4f} mV, full fit {x400 @ quad.coef:.4f} mV, "
      f"fit to T <= 200 °C only {x400 @ q_low.coef:.4f} mV")
print(f"extrapolation error {abs(x400 @ q_low.coef - E[-1])*1000:.0f} µV vs residual SD of the fit {q_low.sigma*1000:.0f} µV")

voltage at 400 °C: measured 16.0490 mV, full fit 16.0461 mV, fit to T <= 200 °C only 15.9146 mV
extrapolation error 134 µV vs residual SD of the fit 10 µV


Within its own range the restricted fit is as good as the full one, but at 400 °C it misses by many times
its residual scatter. The curvature coefficient is poorly determined from the lower half alone, and its
error grows like $T^2$ outside that range. Extrapolation amplifies uncertainty; calibrate over the full
range of use.

## Exercise 3

Linearise $k = A e^{-E_a/RT}$ as $\ln k$ vs $1/T$ and fit it with `linear_lstsq`. Why do the
   weights of the points change when you transform the data? (Preview of notebook 09.)

In [4]:
R = 8.314
T = np.linspace(300, 400, 11)
k_true = 1e7 * np.exp(-50000/(R*T))
sigma = 0.02 * k_true.max()                        # constant ABSOLUTE error (e.g. detector noise)
rng = np.random.default_rng(8)
estimates = {"ln k, equal weights": [], "ln k, weights k²/σ²": [], "nonlinear fit to k": []}
for _ in range(500):
    k = k_true + rng.normal(0, sigma, T.size)
    if np.any(k <= 0):
        continue
    X = np.column_stack([np.ones_like(T), -1/(R*T)])
    estimates["ln k, equal weights"].append(fitting.linear_lstsq(X, np.log(k)).coef[1])
    estimates["ln k, weights k²/σ²"].append(fitting.linear_lstsq(X, np.log(k), weights=k**2).coef[1])
    estimates["nonlinear fit to k"].append(fitting.levenberg_marquardt(
        lambda T, lnA, Ea: np.exp(lnA - Ea/(R*T)), T, k, [16, 50000]).coef[1])
for name, v in estimates.items():
    print(f"{name:<22}: Ea = {np.mean(v)/1000:6.2f} ± {np.std(v)/1000:.2f} kJ/mol (true 50.00)")

ln k, equal weights   : Ea =  46.48 ± 6.70 kJ/mol (true 50.00)
ln k, weights k²/σ²   : Ea =  48.45 ± 1.73 kJ/mol (true 50.00)
nonlinear fit to k    : Ea =  49.96 ± 1.64 kJ/mol (true 50.00)


Taking logarithms changes the error structure: a constant absolute error $\sigma$ in $k$ becomes an error
$\sigma/k$ in $\ln k$ - large for the small rate constants at low temperature. Fitting $\ln k$ with equal
weights therefore trusts the noisiest points as much as the best ones, and the estimate of $E_a$ becomes
much less precise. Weights $k^2/\sigma^2$ (the inverse variance of $\ln k$) restore the balance, and the
nonlinear fit to $k$ itself does so automatically. Linearise for starting values; fit the original model.

## Exercise 4

**Implement it yourself:** write the Huber IRLS loop: compute residuals $r$, a robust scale $s = \mathrm{MAD}/0.6745$, weights $w = \min(1, 1.345/|r/s|)$; refit by weighted least squares; repeat until the coefficients stop changing. Compare with `fitting.huber_irls`.

In [5]:
E_bad = E.copy(); E_bad[-1] -= 0.8
coef = np.linalg.lstsq(X2, E_bad, rcond=None)[0]
for it in range(1, 100):
    r = E_bad - X2 @ coef
    s = np.median(np.abs(r - np.median(r))) / 0.6745
    w = np.minimum(1.0, 1.345 / np.maximum(np.abs(r / s), 1e-12))
    sw = np.sqrt(w)
    new = np.linalg.lstsq(X2 * sw[:, None], E_bad * sw, rcond=None)[0]
    if np.max(np.abs(new - coef)) < 1e-12 * (1 + np.max(np.abs(coef))):
        coef = new; break
    coef = new
lib = fitting.huber_irls(X2, E_bad)
print(f"converged after {it} iterations; max difference from the library {np.max(np.abs(coef - lib.coef)):.1e}")
print("weights:", np.round(w, 2))

converged after 52 iterations; max difference from the library 6.8e-11
weights: [1.   1.   1.   1.   1.   1.   1.   0.73 0.04]


Each pass computes residuals from the current fit, turns them into weights (full weight within 1.345
robust standard deviations, decreasing beyond), and refits by weighted least squares. The faulty reading
ends with a weight near zero - the same result as the library function.